# PiyaNota: clef and key-signature classifier (Kaggle training notebook)

**What this notebook makes.** A small image classifier that looks at the *start* of one staff
(the clef and the key signature right after it) and answers two questions:

1. **Which clef?** `G` (treble), `G8vb` (treble with a small "8" underneath, used for Tenor),
   `F` (bass), or `other` (C clefs, percussion, 8va clefs: all out of scope for PiyaNota).
2. **Which key signature?** A signed count: `-3` means three flats, `+2` means two sharps, `0` means none.

Stage 3 will use this answer instead of the CRNN's clef guess (BUG-011: the CRNN says `clef.C-L3`
almost every time, confidently). The design, the reasons for not using YOLO, and the exact Stage 3
integration are in `docs/decisions/piya-106-clef-keysig-detector-plan.md`. Read that first.

**Training data (and what is NOT used).**
* DeepScoresV2 dense (DS2): real engraved pages with clef and key-signature boxes. Re-split **by piece**
  because the official DS2 train and test sets share 171 of 178 pieces (in different fonts).
* Synthetic 4-staff choir systems rendered with Verovio in 5 music fonts (Leland is MuseScore 4's font).
* **Never used for anything in here:** `piyanota_dataset` (including Flashlight), DSMN, Both Sides Now.
  They are TEST-only. Do not upload them to this notebook.

**Input the model sees.** A *binarised* (black and white) crop, made the same way Stage 1 makes its
native-resolution strip source, but taken **before** Stage 1's non-staff-ink mask (the mask can erase the
small "8" under a Tenor clef). Every training image goes through the same threshold step, so the model
never trains on grey pixels it will not see at inference. Inputs are a fixed 96 x 128 pixels, so there is
no padding and no "true length" problem: a prediction does not depend on what else is in the batch.

**How to run (short version).** Do the setup checklist in the next cell once, set `SMOKE_TEST = True` and
run everything (about 5 minutes) to check it works, then set `SMOKE_TEST = False` and use
**Save Version -> Save & Run All (Commit)**. The full run should take roughly 45-60 minutes on a T4 GPU (most of it is CPU-side image preparation).
At the end, the last cell tells you exactly which files to download and where they go in the repo.

## Step 0. One-time Kaggle setup (read before running anything)

**A. Upload DeepScoresV2 dense as a private Kaggle dataset** (only once; about 0.6 GB after zipping):
1. On your PC, open `D:\PiyaNota\datasets\ds2_dense\`.
2. Make a new folder, for example `D:\ds2_upload\ds2_dense\`, and copy in **only**:
   * the folder `images\` (about 1700 PNG files),
   * `deepscores_train.json`,
   * `deepscores_test.json`.

   Do **not** copy `instance\` or `segmentation\` (not needed, and they add about 250 MB).
3. Right-click `D:\ds2_upload\ds2_dense` -> *Send to* -> *Compressed (zipped) folder*.
4. On kaggle.com: **Datasets -> New Dataset**. Title: `piyanota-ds2-dense`. Drag the zip in.
   Visibility: **Private**. Click **Create**. Wait until Kaggle says processing is finished
   (it unzips the file for you).

**B. Attach it to this notebook.** In the notebook editor, right-hand panel -> **Input** ->
**+ Add Input** -> *Your Work* / *Datasets* -> search `piyanota-ds2-dense` -> **Add**.
You do not need to type the path: the code below searches `/kaggle/input` for `deepscores_train.json`.

**C. Notebook settings** (right-hand panel -> *Session options* / *Settings*):
* **Accelerator: GPU T4 x2** (or a single T4; only one GPU is used). The run was planned for a T4; another
  GPU type changes the numerical results slightly, so write down which one you used. Not "None": the
  notebook stops with an error without a GPU.
* **Internet: On** (needed once to `pip install` Verovio and the SVG renderer).
* **Persistence:** not needed.

**D. Nothing else to upload.** Verovio ships its own music fonts; the synthetic scores are generated here.

## Step 1. Settings

The only setting you normally change is `SMOKE_TEST`:
* `True`: tiny run (a few DS2 pages, a few dozen synthetic systems, 2 epochs). Use it to check the setup.
  Its weights are useless and are marked `smoke_test=True` in the metadata.
* `False`: the real run.

`SEED` fixes every random choice (data split, synthetic scores, augmentation, weight init, batch order).
Running twice with the same seed on the same Kaggle image should give the same numbers. Leave it alone
unless you are deliberately doing a second-seed check (then report both runs as a range).

In [ ]:
import os
import sys
import json
import math
import time
import hashlib
import random
import re
import io
import copy
import platform
import subprocess
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

SMOKE_TEST = False        # <-- set True for the quick check run, False for the real run
SEED = 20260926

SMOKE_TEST = SMOKE_TEST or os.environ.get("CK_SMOKE") == "1"
LOCAL_MODE = not Path("/kaggle/working").exists()
WORK = Path(os.environ.get("CK_WORK", "ck_working")) if LOCAL_MODE else Path("/kaggle/working")
INPUT_ROOT = Path(os.environ.get("CK_INPUT", "ck_input")) if LOCAL_MODE else Path("/kaggle/input")
WORK.mkdir(parents=True, exist_ok=True)

# Data sizes
N_SYNTH_SYSTEMS = 24 if SMOKE_TEST else 3000     # synthetic 4-staff systems (x4 staves each)
MAX_DS2_IMAGES = 24 if SMOKE_TEST else None      # None = all DS2 pages
# Training
EPOCHS = 2 if SMOKE_TEST else 24
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
# Worker processes. Windows cannot fork, so local runs there stay single-process.
N_PROC = 1 if os.name == "nt" else min(4, os.cpu_count() or 1)
NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)

# Splits: every split is decided by a hash of the piece (DS2) or system (synthetic) id, never by page,
# so no piece can appear in two splits in two different fonts.
TRAIN_BUCKETS = 70        # hash % 100 <  70 -> train
VAL_BUCKETS = 15          # 70 <= hash % 100 < 85 -> val, the rest -> test
DS2_UNSEEN_FONT = "gonville"          # DS2 font kept out of train/val entirely (test only)
SYNTH_TRAIN_FONTS = ("Leland", "Bravura", "Leipzig", "Petaluma")
SYNTH_UNSEEN_FONT = "Gootville"       # Verovio font kept out of train/val entirely (test only)
SYNTH_UNSEEN_FONT_PERCENT = 15

# Pre-registered decision rules (fixed BEFORE seeing any result; do not tune them after a run)
CLEF_SELECTIVE_TARGET = 0.995   # confident clef answers must be >= 99.5% right on every val source
KEY_SELECTIVE_TARGET = 0.99     # confident key answers must be >= 99% right on every val source
MIN_SELECTED_FOR_THRESHOLD = 20
G8VB_MIN_RECALL = 0.95          # G8vb is only reported as its own class if val recall >= 95% ...
G8VB_MAX_G_CONFUSION = 0.02     # ... and <= 2% of plain G clefs are called G8vb
G8VB_MIN_SUPPORT = 30           # a val source needs >= 30 true G8vb staves to count as evidence

print(f"LOCAL_MODE={LOCAL_MODE} SMOKE_TEST={SMOKE_TEST} WORK={WORK} INPUT_ROOT={INPUT_ROOT}")
print(f"N_PROC={N_PROC} NUM_WORKERS={NUM_WORKERS}")

## Step 2. Install Verovio and the SVG renderer

Verovio turns a music description (MEI) into an SVG picture; `resvg-py` turns the SVG into pixels.
Versions are pinned so a re-run renders identical images. If this cell fails with a network error,
Internet is off: turn it on in the settings panel and run the cell again.

In [ ]:
VEROVIO_VERSION = "6.3.0"
RESVG_VERSION = "0.5.0"
if not LOCAL_MODE:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"verovio=={VEROVIO_VERSION}", f"resvg-py=={RESVG_VERSION}"],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode != 0:
        raise RuntimeError("pip install failed. Turn Internet ON (Settings panel) and re-run this cell.")

import verovio
import resvg_py

_tk_probe = verovio.toolkit()
print("verovio", _tk_probe.getVersion())
del _tk_probe

## Step 3. Imports and determinism

These switches make the GPU choose deterministic algorithms (the PIYA-106 CRNN runs showed that a seed
alone does not make GPU training repeatable). Every random draw in the data pipeline also comes from a
generator seeded by `(SEED, epoch, sample index)`, so it does not matter which worker process handles which
sample. The cell stops with an error if no GPU is attached on Kaggle.

In [ ]:
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from concurrent.futures import ProcessPoolExecutor


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
cv2.setNumThreads(1)

FORCE_CPU = os.environ.get("CK_FORCE_CPU") == "1"   # local CPU-only checks; never set on Kaggle
DEVICE = torch.device("cpu") if FORCE_CPU else torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not LOCAL_MODE and DEVICE.type != "cuda":
    raise RuntimeError("No GPU attached. Settings panel -> Accelerator -> GPU T4, then restart and run all.")

VERSIONS = {
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(), "opencv": cv2.__version__, "numpy": np.__version__,
    "verovio": verovio.toolkit().getVersion(), "resvg_py": RESVG_VERSION,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
}
print(json.dumps(VERSIONS, indent=1))

## Step 4. The shared crop-and-model module (`clef_key_core.py`)

This cell writes one Python file. It defines **exactly** how a staff is cut out and resized, and the
network. Stage 3 must import this same file unchanged at inference: if the crop at inference differs even
slightly from the crop in training, the model's accuracy numbers stop meaning anything.
You will download this file at the end and put it in the repo. Do not edit it inside the notebook unless
you also bump `CROP_SPEC_VERSION` (that makes old weights refuse to load).

In [ ]:
CORE_SRC = r'''"""Clef and key-signature crop classifier: the shared crop contract and the network.

This file is the single source of truth for BOTH training (the Kaggle notebook
training/piyanota-clef-keysig-kaggle.ipynb writes this exact text to
/kaggle/working/clef_key_core.py) and inference (Stage 3 must import this file
unchanged). The crop is defined in staff spaces, so it does not depend on image
resolution. Changing any constant below changes what the network sees and
invalidates trained weights: bump CROP_SPEC_VERSION when you do.

Input contract (decided 2026-09-26, see docs/decisions/piya-106-clef-keysig-detector-plan.md):
the page passed to make_input() is BINARY, ink = 0 and paper = 255, produced the
way Stage 1 builds its native strip source (adaptive Gaussian threshold at native
resolution) but taken BEFORE the non-staff-ink mask. The mask keeps only ink
attached to the staff, and the small "8" under a treble-8vb clef is not attached,
so the masked page can lose exactly the detail this model exists to see.

The network input is a fixed INPUT_H x INPUT_W image, so there is no padding and
the batch composition cannot change a prediction (BatchNorm runs in eval mode).
"""
from __future__ import annotations

import math
from typing import Optional, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn

CROP_SPEC_VERSION: int = 1

# Geometry of the crop, in staff spaces (sp = distance between two staff lines).
PX_PER_SPACE: int = 8            # output resolution; the "8" of treble-8vb is ~1.2 sp tall -> ~10 px
CROP_LEFT_SPACES: float = 1.0    # context left of the first staff-line column (bracket, barline)
CROP_RIGHT_SPACES: float = 15.0  # clef (~3 sp) + up to 7 key accidentals (~1 sp each) + slack
CROP_ABOVE_SPACES: float = 4.0   # a G clef rises ~2 sp above the top line
CROP_BELOW_SPACES: float = 4.0   # the treble-8vb "8" sits ~1.5-3 sp below the bottom line
STAFF_SPACES: float = 4.0        # five lines = four spaces
INPUT_W: int = int(round((CROP_LEFT_SPACES + CROP_RIGHT_SPACES) * PX_PER_SPACE))                  # 128
INPUT_H: int = int(round((CROP_ABOVE_SPACES + STAFF_SPACES + CROP_BELOW_SPACES) * PX_PER_SPACE))  # 96

# Gaussian pre-blur before down-sampling, as a fraction of the down-scale factor, so thin
# strokes are area-averaged instead of aliased away (same idea as Stage 1's strip source blur).
ANTIALIAS_SIGMA_PER_DOWNSCALE: float = 0.4

# staff_left_edge(): a column is "inside the staff" once at least EDGE_MIN_LINES of the five
# lines carry uninterrupted ink for EDGE_MIN_RUN_SPACES staff spaces to its right.
EDGE_MIN_RUN_SPACES: float = 2.0
EDGE_MIN_LINES: int = 3
EDGE_LINE_TOL_SPACES: float = 0.15

CLEF_CLASSES: Tuple[str, ...] = ("G", "G8vb", "F", "other")
KEY_MIN: int = -7   # 7 flats
KEY_MAX: int = 7    # 7 sharps
KEY_CLASSES: Tuple[int, ...] = tuple(range(KEY_MIN, KEY_MAX + 1))
KEY_IGNORE_INDEX: int = -100


def key_to_index(key: int) -> int:
    """Signed key (-7 = 7 flats ... +7 = 7 sharps) to class index."""
    if not KEY_MIN <= key <= KEY_MAX:
        raise ValueError(f"key {key} outside [{KEY_MIN}, {KEY_MAX}]")
    return key - KEY_MIN


def index_to_key(index: int) -> int:
    """Class index to signed key."""
    return int(index) + KEY_MIN


def staff_left_edge(binary: np.ndarray, top_line_y: float, bottom_line_y: float,
                    search_from_x: int = 0) -> Optional[int]:
    """Finds the first column of the staff lines (where the clef zone starts).

    Args:
        binary: Page, ink < 128.
        top_line_y: Centre row of the top staff line.
        bottom_line_y: Centre row of the bottom staff line.
        search_from_x: Ignore columns left of this.

    Returns:
        Column index, or None if no column has enough continuous line ink.
    """
    ink = binary < 128
    h, w = ink.shape
    sp = (bottom_line_y - top_line_y) / STAFF_SPACES
    if sp <= 0:
        raise ValueError("bottom_line_y must be below top_line_y")
    run = max(2, int(round(EDGE_MIN_RUN_SPACES * sp)))
    tol = max(1, int(round(EDGE_LINE_TOL_SPACES * sp)))
    if run >= w:
        return None
    hits = np.zeros(w - run + 1, dtype=np.int32)
    for k in range(5):
        y = int(round(top_line_y + k * sp))
        lo, hi = max(0, y - tol), min(h, y + tol + 1)
        if lo >= hi:
            continue
        row = ink[lo:hi].any(axis=0).astype(np.int32)
        cs = np.concatenate([[0], np.cumsum(row)])
        hits += ((cs[run:] - cs[:-run]) == run).astype(np.int32)
    cand = np.nonzero(hits[search_from_x:] >= EDGE_MIN_LINES)[0]
    return int(cand[0] + search_from_x) if len(cand) else None


def warp_staff_region(img: np.ndarray, x0: float, top_line_y: float, bottom_line_y: float,
                      px_per_space: float, left_sp: float, right_sp: float,
                      above_sp: float, below_sp: float, *, dx_spaces: float = 0.0,
                      dy_spaces: float = 0.0, scale: float = 1.0,
                      angle_deg: float = 0.0) -> np.ndarray:
    """Resamples a staff-aligned window of a uint8 image (paper = 255) to px_per_space.

    The window is anchored at (x0, staff centre) and measured in staff spaces. dx/dy/scale/
    angle perturb the window (training augmentation and robustness to geometry error);
    at inference they are all neutral.

    Returns:
        uint8 array (round((above+4+below)*pps), round((left+right)*pps)).
    """
    sp = (bottom_line_y - top_line_y) / STAFF_SPACES
    if sp <= 0:
        raise ValueError("bottom_line_y must be below top_line_y")
    out_w = int(round((left_sp + right_sp) * px_per_space))
    out_h = int(round((above_sp + STAFF_SPACES + below_sp) * px_per_space))
    sp_eff = sp * scale
    s = sp_eff / px_per_space                      # source px per output px
    ax = x0 + dx_spaces * sp
    ay = 0.5 * (top_line_y + bottom_line_y) + dy_spaces * sp
    ox0 = -left_sp * sp_eff + 0.5 * s
    oy0 = -(above_sp + STAFF_SPACES / 2.0) * sp_eff + 0.5 * s
    th = math.radians(angle_deg)
    c, sn = math.cos(th), math.sin(th)
    m = np.array([[c * s, -sn * s, ax + c * ox0 - sn * oy0],
                  [sn * s, c * s, ay + sn * ox0 + c * oy0]], dtype=np.float64)

    # Work on the bounding region only (pages are up to 4032 px), padded for the blur.
    corners = np.array([[0, 0, 1], [out_w, 0, 1], [0, out_h, 1], [out_w, out_h, 1]], dtype=np.float64)
    src = corners @ m.T
    sigma = ANTIALIAS_SIGMA_PER_DOWNSCALE * s if s > 1.0 else 0.0
    pad = int(math.ceil(3 * sigma)) + 2
    rx0 = int(math.floor(src[:, 0].min())) - pad
    ry0 = int(math.floor(src[:, 1].min())) - pad
    rx1 = int(math.ceil(src[:, 0].max())) + pad
    ry1 = int(math.ceil(src[:, 1].max())) + pad
    h, w = img.shape[:2]
    region = np.full((ry1 - ry0, rx1 - rx0), 255, dtype=np.uint8)
    sx0, sy0, sx1, sy1 = max(0, rx0), max(0, ry0), min(w, rx1), min(h, ry1)
    if sx0 < sx1 and sy0 < sy1:
        region[sy0 - ry0:sy1 - ry0, sx0 - rx0:sx1 - rx0] = img[sy0:sy1, sx0:sx1]
    if sigma > 0:
        region = cv2.GaussianBlur(region, (0, 0), sigma)
    m_local = m.copy()
    m_local[0, 2] -= rx0
    m_local[1, 2] -= ry0
    return cv2.warpAffine(region, m_local, (out_w, out_h),
                          flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=255)


def make_input(binary_page: np.ndarray, x0: float, top_line_y: float, bottom_line_y: float, *,
               dx_spaces: float = 0.0, dy_spaces: float = 0.0, scale: float = 1.0,
               angle_deg: float = 0.0) -> np.ndarray:
    """The network input for one staff: float32 (INPUT_H, INPUT_W) in [0, 1], ink = 1.

    Args:
        binary_page: Binary page, ink = 0, paper = 255 (see module docstring: UNMASKED).
        x0: First column of the staff lines (staff_left_edge()).
        top_line_y, bottom_line_y: Centre rows of the outer staff lines.
    """
    if binary_page.ndim != 2 or binary_page.dtype != np.uint8:
        raise ValueError("binary_page must be a 2-D uint8 array")
    crop = warp_staff_region(binary_page, x0, top_line_y, bottom_line_y, PX_PER_SPACE,
                             CROP_LEFT_SPACES, CROP_RIGHT_SPACES, CROP_ABOVE_SPACES,
                             CROP_BELOW_SPACES, dx_spaces=dx_spaces, dy_spaces=dy_spaces,
                             scale=scale, angle_deg=angle_deg)
    return (255.0 - crop.astype(np.float32)) / 255.0


def _conv_block(c_in: int, c_out: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class ClefKeyNet(nn.Module):
    """Small two-head CNN: clef class and signed key-signature count.

    The flattened 6x8 map (not a global average pool) keeps horizontal position, which the
    key head needs to count accidentals and to separate the key signature from the first
    note's accidentals.
    """

    def __init__(self, width: int = 32, dropout: float = 0.3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            _conv_block(1, width), _conv_block(width, 2 * width),
            _conv_block(2 * width, 4 * width), _conv_block(4 * width, 4 * width),
        )
        self.reduce = nn.Sequential(nn.Conv2d(4 * width, 64, 1, bias=False),
                                    nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        flat = 64 * (INPUT_H // 16) * (INPUT_W // 16)
        self.trunk = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(flat, 256),
                                   nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.clef_head = nn.Linear(256, len(CLEF_CLASSES))
        self.key_head = nn.Linear(256, len(KEY_CLASSES))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        f = self.trunk(self.reduce(self.features(x)))
        return self.clef_head(f), self.key_head(f)


@torch.no_grad()
def predict_probs(model: ClefKeyNet, inputs: np.ndarray, clef_temperature: float,
                  key_temperature: float, device: Optional[torch.device] = None
                  ) -> Tuple[np.ndarray, np.ndarray]:
    """Temperature-scaled class probabilities for a batch of make_input() arrays (N, H, W)."""
    device = device or next(model.parameters()).device
    model.eval()
    x = torch.from_numpy(np.ascontiguousarray(inputs, dtype=np.float32)).unsqueeze(1).to(device)
    clef_logits, key_logits = model(x)
    return (torch.softmax(clef_logits / clef_temperature, dim=1).cpu().numpy(),
            torch.softmax(key_logits / key_temperature, dim=1).cpu().numpy())
'''

core_path = WORK / "clef_key_core.py"
core_path.write_bytes(CORE_SRC.encode("utf-8"))   # bytes: no newline conversion, so the sha256 matches
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import importlib
import clef_key_core as ck
ck = importlib.reload(ck)
CORE_SHA256 = hashlib.sha256(CORE_SRC.encode("utf-8")).hexdigest()
print("clef_key_core.py written; sha256", CORE_SHA256[:16], "input", ck.INPUT_H, "x", ck.INPUT_W,
      "crop spec v", ck.CROP_SPEC_VERSION)

## Step 5. Find the DeepScoresV2 files

Searches every attached dataset for `deepscores_train.json` and `deepscores_test.json` and expects an
`images` folder next to them. If this fails, go back to Step 0 B (the dataset is not attached) or check
that the zip contained `images/` and both JSON files.

In [ ]:
def find_one(name: str) -> Path:
    hits = sorted(INPUT_ROOT.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Attach the piyanota-ds2-dense dataset "
                                "(Step 0 B).")
    if len(hits) > 1:
        print(f"WARNING: several {name} found, using {hits[0]}")
    return hits[0]


DS2_TRAIN_JSON = find_one("deepscores_train.json")
DS2_TEST_JSON = find_one("deepscores_test.json")
DS2_IMAGE_DIR = DS2_TRAIN_JSON.parent / "images"
if not DS2_IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Expected the DS2 images folder at {DS2_IMAGE_DIR}")
print(DS2_TRAIN_JSON, DS2_TEST_JSON, DS2_IMAGE_DIR, sum(1 for _ in DS2_IMAGE_DIR.glob("*.png")), "png files")

## Step 6. Read the DS2 labels for every staff

For each staff box on each page we look for the **opening** clef (the leftmost clef near the start of that
staff) and the key-signature accidentals right after it. Rules (they are also written in the plan doc):
* `clefG` with a DS2 `clef8` box just **below** it -> `G8vb`. An "8" above, a "15", an F clef with an "8",
  C clefs and percussion clefs -> `other`.
* Key = number of `keySharp` (positive) or `keyFlat` (negative) boxes between the clef's right edge and
  10 staff spaces further right, vertically on this staff. If `keyNatural` boxes are there (a cancelled
  key), or sharps and flats are mixed, or there are more than 7, the key label is set to "ignore" (the
  clef label is still used).
* Staves with no opening clef are skipped (they are counted and printed).

Only DS2's own category names are used (`deepscores` annotation set); the duplicated `muscima++`
category ids in the same files are ignored.

In [ ]:
PIECE_RE = re.compile(r"^lg-(.+?)-aug-([a-z0-9]+)-")
CLEF_CATS = {"clefG": "G", "clefF": "F", "clefCAlto": "C", "clefCTenor": "C", "clefUnpitchedPercussion": "perc"}
WANTED = set(CLEF_CATS) | {"clef8", "clef15", "keySharp", "keyFlat", "keyNatural", "staff"}
KEY_IGNORE = ck.KEY_IGNORE_INDEX


def _small_ink_near(clef_box, boxes, sp, where):
    """True if any small box (a DS2 clef8/clef15) sits directly below/above the clef box."""
    cx0, cy0, cx1, cy1 = clef_box
    for b in boxes:
        bx = 0.5 * (b[0] + b[2])
        if not (cx0 - 0.5 * sp <= bx <= cx1 + 0.5 * sp):
            continue
        if where == "below" and cy1 - 1.0 * sp <= b[1] <= cy1 + 1.5 * sp:
            return True
        if where == "above" and cy0 - 1.5 * sp <= b[3] <= cy0 + 1.0 * sp:
            return True
    return False


def ds2_staff_records(json_path: Path, tag: str):
    with open(json_path, "r", encoding="utf-8") as f:
        d = json.load(f)
    name_by_id = {k: v["name"] for k, v in d["categories"].items() if v["annotation_set"] == "deepscores"}
    stats = Counter()
    records = []
    for im in d["images"]:
        m = PIECE_RE.match(im["filename"])
        if not m:
            stats["bad_filename"] += 1
            continue
        piece, font = m.group(1), m.group(2)
        objs = defaultdict(list)
        for aid in im["ann_ids"]:
            a = d["annotations"][aid]
            names = {name_by_id[c] for c in a["cat_id"] if c in name_by_id}
            for n in names & WANTED:
                objs[n].append([float(v) for v in a["a_bbox"]])
        clefs = [(n, b) for n in CLEF_CATS for b in objs[n]]
        for si, sb in enumerate(sorted(objs["staff"], key=lambda b: b[1])):
            x0, y0, x1, y1 = sb
            sp = (y1 - y0) / 4.0
            if sp <= 2:
                stats["tiny_staff"] += 1
                continue
            cand = [(n, b) for n, b in clefs
                    if b[0] < x0 + 5 * sp and b[2] > x0 - 1 * sp and y0 - sp <= 0.5 * (b[1] + b[3]) <= y1 + sp]
            if not cand:
                stats["no_opening_clef"] += 1
                continue
            cname, cb = min(cand, key=lambda t: t[1][0])
            base = CLEF_CATS[cname]
            below8 = _small_ink_near(cb, objs["clef8"], sp, "below")
            above8 = _small_ink_near(cb, objs["clef8"], sp, "above")
            any15 = (_small_ink_near(cb, objs["clef15"], sp, "below")
                     or _small_ink_near(cb, objs["clef15"], sp, "above"))
            if base == "G" and below8 and not above8 and not any15:
                clef = "G8vb"
            elif base == "G" and not (below8 or above8 or any15):
                clef = "G"
            elif base == "F" and not (below8 or above8 or any15):
                clef = "F"
            else:
                clef = "other"

            def _in_key_zone(b):
                cx, cy = 0.5 * (b[0] + b[2]), 0.5 * (b[1] + b[3])
                return cb[2] - 0.3 * sp <= cx <= cb[2] + 10 * sp and y0 - 2 * sp <= cy <= y1 + 2 * sp

            ns = sum(_in_key_zone(b) for b in objs["keySharp"])
            nf = sum(_in_key_zone(b) for b in objs["keyFlat"])
            nn_ = sum(_in_key_zone(b) for b in objs["keyNatural"])
            if nn_ or (ns and nf) or max(ns, nf) > ck.KEY_MAX:
                key, key_note = KEY_IGNORE, "ambiguous"
                stats["key_ignored"] += 1
            else:
                key, key_note = (ns if ns else -nf), "ok"
            stats[f"clef_{clef}"] += 1
            records.append({"source": "ds2", "json": tag, "filename": im["filename"], "piece": piece,
                            "font": font, "staff_idx": si, "bbox": sb, "clef": clef, "clef_raw": cname,
                            "key": key, "key_note": key_note})
    del d
    return records, stats


ds2_records = []
for path, tag in ((DS2_TRAIN_JSON, "official_train"), (DS2_TEST_JSON, "official_test")):
    t0 = time.time()
    recs, st = ds2_staff_records(path, tag)
    ds2_records += recs
    print(tag, len(recs), "staves with an opening clef;", dict(st), f"({time.time() - t0:.0f}s)")

## Step 7. Split DS2 by piece (not by page)

DS2 renders each piece in up to five fonts, and the official DS2 test set shares almost all its pieces
with the official train set. A page-level split would therefore put "the same music in a different font"
into both train and test, and the test score would be inflated. Here the split is a hash of the piece id:
~70% of pieces train, ~15% val, ~15% test. The `gonville` font is never trained on: its pages are kept
only for test pieces, as an "unseen font" test.

In [ ]:
def bucket(key: str) -> int:
    return int(hashlib.sha1(key.encode("utf-8")).hexdigest(), 16) % 100


def split_for_bucket(b: int) -> str:
    return "train" if b < TRAIN_BUCKETS else ("val" if b < TRAIN_BUCKETS + VAL_BUCKETS else "test")


for r in ds2_records:
    s = split_for_bucket(bucket("ds2:" + r["piece"]))
    if r["font"] == DS2_UNSEEN_FONT:
        s = "test_unseen_font" if s == "test" else None
    r["split"] = s
ds2_records = [r for r in ds2_records if r["split"] is not None]

official = defaultdict(set)
for r in ds2_records:
    official[r["json"]].add(r["piece"])
print("pieces in BOTH official DS2 train and test:",
      len(official["official_train"] & official["official_test"]), "(this is why we re-split)")

if MAX_DS2_IMAGES is not None:  # smoke test: a few pages from every split
    keep = set()
    by_split = defaultdict(list)
    for fn, s in sorted({(r["filename"], r["split"]) for r in ds2_records}):
        by_split[s].append(fn)
    for s, fns in by_split.items():
        keep |= set(sorted(fns, key=lambda x: bucket("pick:" + x))[:max(1, MAX_DS2_IMAGES // 4)])
    ds2_records = [r for r in ds2_records if r["filename"] in keep]

piece_split = defaultdict(set)
for r in ds2_records:
    piece_split[r["piece"]].add("test" if r["split"].startswith("test") else r["split"])
assert all(len(v) == 1 for v in piece_split.values()), "a piece landed in two splits"
print(Counter(r["split"] for r in ds2_records))

## Step 8. Binarisation (matches Stage 1) and the stored crop format

`binarize_like_stage1` repeats Stage 1's strip-source steps (divide by a smooth background, light blur,
adaptive Gaussian threshold with offset C = 10). Stage 1's threshold window is fixed in pixels
(31 px at 1600 px page width, scaled with the page), which is about 2-3 staff spaces on normal scores.
Here the window is set in staff spaces: 2.5 for validation/test, randomly 1.8-3.5 in training so the model
is not tied to one exact value.

To save memory, every staff is first stored as a **grey** crop at 12 pixels per staff space (the grey
version is kept so the camera-style damage in Step 11 can be applied *before* binarising, the way a real
photo is damaged before Stage 1 binarises it). `x0` (where the staff lines start) is found with the same
`staff_left_edge()` function Stage 3 will use.

In [ ]:
STORE_PPS = 12
STORE_LEFT, STORE_RIGHT, STORE_ABOVE, STORE_BELOW = 4.0, 19.0, 7.0, 7.0
STORE_W = int(round((STORE_LEFT + STORE_RIGHT) * STORE_PPS))                   # 276
STORE_H = int(round((STORE_ABOVE + ck.STAFF_SPACES + STORE_BELOW) * STORE_PPS))  # 216
# Pixel-centre rows/cols of the top line, bottom line and staff start inside a stored crop
STORE_TOP = STORE_ABOVE * STORE_PPS - 0.5
STORE_BOTTOM = STORE_TOP + ck.STAFF_SPACES * STORE_PPS
STORE_X0 = STORE_LEFT * STORE_PPS - 0.5

CANON_BLOCK_SPACES = 2.5
CANON_C = 10.0
CANON_UPSCALE = 2.0        # val/test crops are binarised at 24 px per staff space (typical phone photo)
BLUR_SPACES = 0.035        # Stage 1: sigma ~1 px at ~30 px staff space
BG_KERNEL_SPACES = 1.5


def _odd(v: float) -> int:
    return max(3, int(round(v)) | 1)


def binarize_like_stage1(gray: np.ndarray, sp_px: float, block_spaces: float = CANON_BLOCK_SPACES,
                         c: float = CANON_C) -> np.ndarray:
    """Background-normalise, blur, adaptive Gaussian threshold. Returns ink = 0, paper = 255."""
    h, w = gray.shape
    small = cv2.resize(gray, (max(1, w // 4), max(1, h // 4)), interpolation=cv2.INTER_AREA)
    k = _odd(BG_KERNEL_SPACES * sp_px / 4)
    bg = cv2.dilate(small, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    bg = cv2.GaussianBlur(bg, (0, 0), max(1.0, 0.5 * sp_px))
    bg = cv2.resize(bg, (w, h), interpolation=cv2.INTER_LINEAR)
    norm = cv2.divide(gray, np.maximum(bg, 1), scale=255)
    norm = cv2.GaussianBlur(norm, (0, 0), max(0.5, BLUR_SPACES * sp_px))
    return cv2.adaptiveThreshold(norm, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
                                 _odd(block_spaces * sp_px), c)


def store_crop(gray_page: np.ndarray, x0: float, top: float, bottom: float):
    """Grey storage crop plus the staff-start column found the way Stage 3 finds it."""
    crop = ck.warp_staff_region(gray_page, x0, top, bottom, STORE_PPS, STORE_LEFT, STORE_RIGHT,
                                STORE_ABOVE, STORE_BELOW)
    edge = ck.staff_left_edge(binarize_like_stage1(crop, STORE_PPS), STORE_TOP, STORE_BOTTOM)
    if edge is None or abs(edge - STORE_X0) > 2 * STORE_PPS:
        return crop, float(STORE_X0), False
    return crop, float(edge), True


def refine_staff_lines(gray: np.ndarray, bbox):
    """Centre rows of the top and bottom staff line inside a DS2 staff box."""
    x0, y0, x1, y1 = bbox
    h = gray.shape[0]
    pad = 4
    ya, yb = max(0, int(y0) - pad), min(h, int(math.ceil(y1)) + pad + 1)
    xa, xb = int(x0 + 0.1 * (x1 - x0)), int(x1 - 0.1 * (x1 - x0))
    frac = (gray[ya:yb, xa:xb] < 128).mean(axis=1)
    rows = np.nonzero(frac > 0.5)[0]
    groups = np.split(rows, np.nonzero(np.diff(rows) > 1)[0] + 1) if len(rows) else []
    if len(groups) == 5:
        c = [ya + float(g.mean()) for g in groups]
        return c[0], c[-1], True
    return y0 + 1.0, y1 - 1.0, False


print("stored crop", STORE_H, "x", STORE_W, "top/bottom/x0", STORE_TOP, STORE_BOTTOM, STORE_X0)

## Step 9. Cut the DS2 staves out of the page images

Reads every needed DS2 page once and stores one grey crop per labelled staff. On the full dataset this
takes a few minutes. The printed counts tell you how often the exact staff-line rows or the staff start
could not be measured (a fallback is used; a few percent is normal).

In [ ]:
def process_ds2_image(job):
    fn, recs = job
    gray = cv2.imread(str(DS2_IMAGE_DIR / fn), cv2.IMREAD_GRAYSCALE)
    out = []
    if gray is None:
        return fn, out
    for r in recs:
        top, bottom, lines_ok = refine_staff_lines(gray, r["bbox"])
        crop, x0s, edge_ok = store_crop(gray, r["bbox"][0], top, bottom)
        out.append((r["_i"], crop, x0s, lines_ok, edge_ok))
    return fn, out


for i, r in enumerate(ds2_records):
    r["_i"] = i
jobs_by_file = defaultdict(list)
for r in ds2_records:
    jobs_by_file[r["filename"]].append(r)
ds2_jobs = sorted(jobs_by_file.items())

t0 = time.time()
DS2_RAW = np.zeros((len(ds2_records), STORE_H, STORE_W), dtype=np.uint8)
missing_files, n_lines_fb, n_edge_fb = [], 0, 0
if N_PROC > 1:
    with ProcessPoolExecutor(max_workers=N_PROC) as ex:
        results = list(ex.map(process_ds2_image, ds2_jobs, chunksize=4))
else:
    results = [process_ds2_image(j) for j in ds2_jobs]
got = np.zeros(len(ds2_records), dtype=bool)
for fn, out in results:
    if not out:
        missing_files.append(fn)
    for i, crop, x0s, lines_ok, edge_ok in out:
        DS2_RAW[i] = crop
        ds2_records[i]["x0_store"] = x0s
        n_lines_fb += not lines_ok
        n_edge_fb += not edge_ok
        got[i] = True
del results
if missing_files:
    print(f"WARNING: {len(missing_files)} DS2 image files could not be read, e.g. {missing_files[:3]}")
keep_idx = np.nonzero(got)[0]
DS2_RAW = DS2_RAW[keep_idx]
ds2_records = [ds2_records[i] for i in keep_idx]
print(f"DS2 crops: {len(ds2_records)} in {time.time() - t0:.0f}s; staff-line fallback {n_lines_fb}, "
      f"staff-start fallback {n_edge_fb}")

## Step 10. Synthetic choir systems with Verovio

DS2 has almost no treble-8vb clefs (about 600 in 53 pieces), no MuseScore 4 font (Leland), and mostly
piano/orchestral layouts. So we also generate 4-staff choir systems:
* each staff gets its clef **independently** (G, G8vb, F, or an "other" clef), so the model cannot guess
  the clef from the staff's position in the system (Flashlight, for example, has Soloist / Soprano / Alto /
  Tenor with no Bass staff);
* staff names ("Soprano", "T.", "Soloist", ...) are chosen independently of the clef, so the name text is
  not a shortcut;
* all four staves share one key (-7..+7), with random notes, accidentals, rests, lyrics, dynamics, time
  signatures shown or hidden (hidden = a later system), different spacing, line widths and sizes;
* fonts Leland, Bravura, Leipzig, Petaluma for train/val/test; **Gootville only in a separate test set**.

Systems are split 80/10/10 by system id. The first 4 rendered systems are shown after this cell so you can
check they look like music.

In [ ]:
SYNTH_CLEFS = {  # name: (MEI shape, line, dis, dis.place, class label, bottom-line diatonic index)
    "G": ("G", 2, None, None, "G", 4 * 7 + 2),        # bottom line E4
    "G8vb": ("G", 2, 8, "below", "G8vb", 3 * 7 + 2),  # sounds an octave lower: bottom line E3
    "F": ("F", 4, None, None, "F", 2 * 7 + 4),        # bottom line G2
    "C3": ("C", 3, None, None, "other", 3 * 7 + 3),   # alto clef, bottom line F3
    "C4": ("C", 4, None, None, "other", 3 * 7 + 1),   # tenor C clef, bottom line D3
    "G8va": ("G", 2, 8, "above", "other", 5 * 7 + 2),
    "F8vb": ("F", 4, 8, "below", "other", 1 * 7 + 4),
}
SYNTH_CLEF_WEIGHTS = {"G": 0.30, "G8vb": 0.25, "F": 0.30, "C3": 0.05, "C4": 0.05, "G8va": 0.03, "F8vb": 0.02}
SYNTH_LABELS = ["Soprano", "Alto", "Tenor", "Bass", "S.", "A.", "T.", "B.", "Sop.", "Ten.", "Soloist",
                "Solo", "Voice", "Choir", "Men", "Women", "S 1", "T 2", "Bar.", "Mezzo", "", "", ""]
SYLLABLES = ["la", "ah", "oh", "sing", "Glo", "ri", "a", "praise", "the", "Lord", "ma", "hal", "le",
             "lu", "ja", "love", "ka", "sa", "ma", "mo", "Pa", "sko", "ng", "ta", "yo"]
PNAMES = "cdefgab"
KEY_VALUES = np.arange(ck.KEY_MIN, ck.KEY_MAX + 1)
KEY_WEIGHTS = np.where(np.abs(KEY_VALUES) <= 4, 1.0, 0.5)
KEY_WEIGHTS = KEY_WEIGHTS / KEY_WEIGHTS.sum()
SYNTH_TARGET_SP = (14.0, 28.0)    # rendered staff space in pixels (phone photos of a page: ~20-40 at native)


def _note_xml(rng, bottom_diatonic, dur, dots, lyric):
    if rng.random() < 0.10:
        return f'<rest dur="{dur}"{dots}/>'
    pos = int(rng.integers(-3, 12))                    # staff position; 0 = bottom line, 8 = top line
    d = bottom_diatonic + pos
    accid = ""
    if rng.random() < 0.12:
        accid = f' accid="{rng.choice(["s", "f", "n"])}"'
    syl = f'<verse n="1"><syl>{rng.choice(SYLLABLES)}</syl></verse>' if lyric else ""
    return f'<note pname="{PNAMES[d % 7]}" oct="{d // 7}" dur="{dur}"{dots}{accid}>{syl}</note>'


def _measure_xml(rng, beats, bottom_diatonic, lyric):
    left, parts = float(beats), []
    while left > 1e-6:
        opts = [(q, dur, dots) for q, dur, dots in ((4, "1", ""), (3, "2", ' dots="1"'), (2, "2", ""),
                                                   (1, "4", ""), (0.5, "8", "")) if q <= left + 1e-6]
        q, dur, dots = opts[int(rng.integers(len(opts)))]
        if q == 0.5:
            parts += [_note_xml(rng, bottom_diatonic, "8", "", lyric) for _ in range(2)]
            left -= 1.0
        else:
            parts.append(_note_xml(rng, bottom_diatonic, dur, dots, lyric))
            left -= q
    return "".join(parts)


def make_system(sys_id: int):
    """One synthetic 4-staff system: (MEI text, per-staff labels, Verovio options, target staff space)."""
    rng = np.random.default_rng([SEED, 7, sys_id])
    names = list(SYNTH_CLEF_WEIGHTS)
    p = np.array([SYNTH_CLEF_WEIGHTS[n] for n in names])
    clefs = [names[i] for i in rng.choice(len(names), size=4, p=p / p.sum())]
    key = int(rng.choice(KEY_VALUES, p=KEY_WEIGHTS))
    sig = "0" if key == 0 else f"{abs(key)}{'s' if key > 0 else 'f'}"
    beats = int(rng.choice([2, 3, 4]))
    meter_vis = "" if rng.random() < 0.5 else ' visible="false"'
    n_measures = int(rng.integers(2, 4))
    lyric_staff = [rng.random() < 0.4 for _ in range(4)]
    staff_defs, labels = [], []
    for n, cname in enumerate(clefs, start=1):
        shape, line, dis, place, cls, _ = SYNTH_CLEFS[cname]
        dis_xml = f' dis="{dis}" dis.place="{place}"' if dis else ""
        lab = SYNTH_LABELS[int(rng.integers(len(SYNTH_LABELS)))]
        lab_xml = f"<label>{lab}</label>" if lab else ""
        staff_defs.append(f'<staffDef n="{n}" lines="5">{lab_xml}<clef shape="{shape}" line="{line}"{dis_xml}/>'
                          f'<keySig sig="{sig}"/><meterSig count="{beats}" unit="4"{meter_vis}/></staffDef>')
        labels.append({"clef_name": cname, "clef": cls, "key": key, "label": lab, "lyrics": lyric_staff[n - 1]})
    measures = []
    for m in range(1, n_measures + 1):
        staves = "".join(f'<staff n="{n}"><layer n="1">'
                         f'{_measure_xml(rng, beats, SYNTH_CLEFS[c][5], lyric_staff[n - 1])}</layer></staff>'
                         for n, c in enumerate(clefs, start=1))
        dyn = ""
        if m == 1 and rng.random() < 0.3:
            dyn = f'<dynam staff="{int(rng.integers(1, 5))}" tstamp="1">{rng.choice(["p", "mf", "f", "mp"])}</dynam>'
        measures.append(f'<measure n="{m}">{staves}{dyn}</measure>')
    symbol = rng.choice(["bracket", "brace", "none"])
    grp_attr = f' symbol="{symbol}"' if symbol != "none" else ""
    mei = ('<?xml version="1.0" encoding="UTF-8"?>'
           '<mei xmlns="http://www.music-encoding.org/ns/mei" meiversion="5.0">'
           '<meiHead><fileDesc><titleStmt><title>s</title></titleStmt><pubStmt/></fileDesc></meiHead>'
           '<music><body><mdiv><score><scoreDef>'
           f'<staffGrp{grp_attr} bar.thru="{"true" if rng.random() < 0.3 else "false"}">'
           + "".join(staff_defs) + '</staffGrp></scoreDef><section>' + "".join(measures)
           + '</section></score></mdiv></body></music></mei>')
    options = {"breaks": "none", "adjustPageHeight": True, "header": "none", "footer": "none",
               "scale": 100, "spacingStaff": int(rng.integers(7, 15)),
               "spacingLinear": float(rng.uniform(0.2, 0.35)), "spacingNonLinear": float(rng.uniform(0.5, 0.7)),
               "staffLineWidth": float(rng.uniform(0.12, 0.25)), "stemWidth": float(rng.uniform(0.15, 0.3)),
               "lyricSize": float(rng.uniform(3.5, 5.0))}
    target_sp = float(rng.uniform(*SYNTH_TARGET_SP))
    return mei, labels, options, target_sp


def find_staves_clean(gray: np.ndarray):
    """Top/bottom line centre rows of each staff in a clean render (row-projection)."""
    frac = (gray < 128).mean(axis=1)
    if frac.max() <= 0:
        return []
    rows = np.nonzero(frac > 0.5 * frac.max())[0]
    groups = np.split(rows, np.nonzero(np.diff(rows) > 1)[0] + 1) if len(rows) else []
    lines = [float(g.mean()) for g in groups]
    staves, i = [], 0
    while i + 5 <= len(lines):
        gaps = np.diff(lines[i:i + 5])
        if gaps.min() > 2 and gaps.max() / gaps.min() <= 1.2:
            staves.append((lines[i], lines[i + 4]))
            i += 5
        else:
            i += 1
    return staves


_TK = None
_SP100 = None


def _toolkit():
    global _TK
    if _TK is None:
        _TK = verovio.toolkit()
    return _TK


def render_gray(mei: str, options: dict, font: str, zoom: float) -> np.ndarray:
    tk = _toolkit()
    tk.setOptions({**options, "font": font})
    if not tk.loadData(mei):
        raise ValueError("verovio could not load the MEI")
    svg = tk.renderToSVG(1)
    png = resvg_py.svg_to_bytes(svg_string=svg, background="#ffffff", zoom=float(zoom))
    return cv2.imdecode(np.frombuffer(bytes(png), np.uint8), cv2.IMREAD_GRAYSCALE)


def staff_space_at_zoom1() -> float:
    """Measured once: staff space in pixels at zoom 1 (Verovio scale 100)."""
    mei, _, opts, _ = make_system(0)
    g = render_gray(mei, opts, "Leipzig", 1.0)
    st = find_staves_clean(g)
    if not st:
        raise RuntimeError("calibration render has no staves")
    return float(np.median([(b - t) / 4.0 for t, b in st]))


def synth_font_and_split(sys_id: int):
    if bucket(f"synthfont:{SEED}:{sys_id}") < SYNTH_UNSEEN_FONT_PERCENT:
        return SYNTH_UNSEEN_FONT, "test_unseen_font"
    rng = np.random.default_rng([SEED, 8, sys_id])
    font = SYNTH_TRAIN_FONTS[int(rng.integers(len(SYNTH_TRAIN_FONTS)))]
    b = bucket(f"synthsplit:{SEED}:{sys_id}")
    return font, ("train" if b < 80 else ("val" if b < 90 else "test"))


def process_synth_system(sys_id: int):
    mei, labels, opts, target_sp = make_system(sys_id)
    font, split = synth_font_and_split(sys_id)
    try:
        gray = render_gray(mei, opts, font, target_sp / SP_ZOOM1)
    except (ValueError, RuntimeError) as e:
        return sys_id, None, f"render_error:{e}"
    staves = find_staves_clean(gray)
    if len(staves) != 4:
        return sys_id, None, f"found_{len(staves)}_staves"
    out = []
    for si, ((top, bottom), lab) in enumerate(zip(staves, labels)):
        x0 = ck.staff_left_edge(gray, top, bottom)
        if x0 is None:
            return sys_id, None, "no_staff_start"
        crop, x0s, edge_ok = store_crop(gray, x0, top, bottom)
        out.append((crop, {"source": "synth", "sys_id": sys_id, "staff_idx": si, "font": font,
                           "split": split, "x0_store": x0s, **lab}))
    return sys_id, out, "ok"


SP_ZOOM1 = staff_space_at_zoom1()
print("staff space at zoom 1:", round(SP_ZOOM1, 2), "px")

t0 = time.time()
if N_PROC > 1:
    with ProcessPoolExecutor(max_workers=N_PROC) as ex:
        synth_results = list(ex.map(process_synth_system, range(N_SYNTH_SYSTEMS), chunksize=8))
else:
    synth_results = [process_synth_system(i) for i in range(N_SYNTH_SYSTEMS)]
synth_status = Counter(s for _, _, s in synth_results)
synth_records, synth_crops = [], []
for _, out, s in synth_results:
    if out:
        for crop, meta in out:
            synth_crops.append(crop)
            synth_records.append(meta)
del synth_results
SYNTH_RAW = np.stack(synth_crops) if synth_crops else np.zeros((0, STORE_H, STORE_W), np.uint8)
del synth_crops
print(f"synthetic: {len(synth_records)} staves from {N_SYNTH_SYSTEMS} systems in {time.time() - t0:.0f}s;",
      dict(synth_status))

import matplotlib
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
for ax, sid in zip(axes, range(4)):
    mei, labs, opts, tsp = make_system(sid)
    fnt, _ = synth_font_and_split(sid)
    ax.imshow(render_gray(mei, opts, fnt, tsp / SP_ZOOM1), cmap="gray")
    ax.set_title(f"{fnt}: " + ", ".join(l["clef_name"] for l in labs) + f" key {labs[0]['key']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(WORK / "preview_synthetic_systems.png", dpi=80)
plt.show()

## Step 11. Put everything together and write the split manifest

From here on DS2 and synthetic staves live in one table, but the **source is kept on every row** and every
number below is reported per source and per split, never blended into one total.
`clef_key_splits.json` records which piece/system went into which split (download it; it proves the test
pieces were never trained on).

In [ ]:
ALL_META = ds2_records + synth_records
RAW = np.concatenate([DS2_RAW, SYNTH_RAW], axis=0)
del DS2_RAW, SYNTH_RAW
CLEF_Y = np.array([ck.CLEF_CLASSES.index(m["clef"]) for m in ALL_META], dtype=np.int64)
KEY_Y = np.array([KEY_IGNORE if m["key"] == KEY_IGNORE else ck.key_to_index(m["key"]) for m in ALL_META],
                 dtype=np.int64)
X0S = np.array([m["x0_store"] for m in ALL_META], dtype=np.float32)
SRC = np.array([m["source"] for m in ALL_META])
SPLIT = np.array([m["split"] for m in ALL_META])

EVAL_SETS = {  # name -> row indices
    "val_ds2": np.nonzero((SRC == "ds2") & (SPLIT == "val"))[0],
    "val_synth": np.nonzero((SRC == "synth") & (SPLIT == "val"))[0],
    "test_ds2_seen_fonts": np.nonzero((SRC == "ds2") & (SPLIT == "test"))[0],
    "test_ds2_unseen_font_gonville": np.nonzero((SRC == "ds2") & (SPLIT == "test_unseen_font"))[0],
    "test_synth_seen_fonts": np.nonzero((SRC == "synth") & (SPLIT == "test"))[0],
    "test_synth_unseen_font_gootville": np.nonzero((SRC == "synth") & (SPLIT == "test_unseen_font"))[0],
}
VAL_SETS = ["val_ds2", "val_synth"]
TRAIN_IDX = np.nonzero(SPLIT == "train")[0]

rows = []
for name, idx in [("train_ds2", TRAIN_IDX[SRC[TRAIN_IDX] == "ds2"]),
                  ("train_synth", TRAIN_IDX[SRC[TRAIN_IDX] == "synth"])] + list(EVAL_SETS.items()):
    cc = Counter(ck.CLEF_CLASSES[c] for c in CLEF_Y[idx])
    rows.append((name, len(idx), *[cc.get(c, 0) for c in ck.CLEF_CLASSES], int((KEY_Y[idx] != KEY_IGNORE).sum())))
print(f"{'set':36s} {'n':>6s} " + " ".join(f"{c:>6s}" for c in ck.CLEF_CLASSES) + "  key_labelled")
for r in rows:
    print(f"{r[0]:36s} {r[1]:6d} " + " ".join(f"{v:6d}" for v in r[2:6]) + f"  {r[6]:6d}")

groups = sorted({("ds2", m["piece"], m["split"]) for m in ds2_records}
                | {("synth", str(m["sys_id"]), m["split"]) for m in synth_records})
manifest = {"seed": SEED, "rule": "ds2: sha1('ds2:'+piece)%100 (<70 train, <85 val, else test; gonville test-only); "
                                  "synth: see synth_font_and_split()",
            "groups": [{"source": s, "group": g, "split": sp} for s, g, sp in groups]}
manifest_text = json.dumps(manifest, sort_keys=True)
SPLIT_MANIFEST_SHA256 = hashlib.sha256(manifest_text.encode("utf-8")).hexdigest()
(WORK / "clef_key_splits.json").write_text(manifest_text, encoding="utf-8")
print("split manifest sha256", SPLIT_MANIFEST_SHA256[:16], "groups", len(groups))

## Step 12. Turning a stored crop into a network input (training damage vs. clean evaluation)

**Training** (`augment=True`), in this order, so it looks like Stage 1 output from a phone photo:
1. camera-style damage on the grey crop: uneven lighting, faded ink, paper tone, blur, low resolution,
   sensor noise, JPEG compression, dust specks (each only sometimes);
2. random resolution (12-30 px per staff space) and random threshold settings, then binarise;
3. sometimes thicker or thinner strokes (the threshold's effect varies between photos);
4. small random shifts of the crop window (staff start +-0.6 staff space, height +-0.3), size +-10%,
   rotation +-1.5 degrees, because Stage 1/2 geometry is never perfect.

**Validation and test** (`augment=False`): no damage, fixed canonical binarisation, no shifts. Separately,
a *stress* copy of each test set applies the training damage with a different seed, so you can see how
much worse the model gets on damaged input. All inputs are the same fixed size, so nothing is padded.

In [ ]:
GEO_DX, GEO_DY, GEO_SCALE, GEO_ANGLE = 0.6, 0.3, (0.9, 1.1), 1.5


def photometric_augment(g: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    g = g.astype(np.float32)
    h, w = g.shape
    if rng.random() < 0.5:   # uneven lighting
        yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
        a = rng.uniform(0, 2 * np.pi)
        ramp = np.cos(a) * xx / w + np.sin(a) * yy / h
        ramp = (ramp - ramp.min()) / (np.ptp(ramp) + 1e-6)
        g = g * (1.0 - rng.uniform(0.0, 0.45) * ramp)
    if rng.random() < 0.3:   # faded ink / low contrast
        g = 255.0 - (255.0 - g) * rng.uniform(0.45, 0.9)
    if rng.random() < 0.3:   # paper tone
        g = g * rng.uniform(0.75, 1.0)
    if rng.random() < 0.4:   # focus blur
        g = cv2.GaussianBlur(g, (0, 0), rng.uniform(0.3, 1.2))
    if rng.random() < 0.3:   # low resolution
        f = rng.uniform(0.4, 0.9)
        small = cv2.resize(g, (max(1, int(w * f)), max(1, int(h * f))), interpolation=cv2.INTER_AREA)
        g = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if rng.random() < 0.5:   # sensor noise
        g = g + rng.normal(0.0, rng.uniform(2.0, 12.0), g.shape).astype(np.float32)
    g = np.clip(g, 0, 255).astype(np.uint8)
    if rng.random() < 0.4:   # JPEG
        ok, enc = cv2.imencode(".jpg", g, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(30, 91))])
        if ok:
            g = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE)
    if rng.random() < 0.2:   # dust specks
        for _ in range(int(rng.integers(3, 30))):
            cv2.circle(g, (int(rng.integers(w)), int(rng.integers(h))), int(rng.integers(0, 2)),
                       int(rng.integers(0, 120)), -1)
    return g


def build_input(raw: np.ndarray, x0_store: float, rng=None) -> np.ndarray:
    """Stored grey crop -> binarised, resampled (INPUT_H, INPUT_W) float32 input (ink = 1)."""
    aug = rng is not None
    g = photometric_augment(raw, rng) if aug else raw
    f = rng.uniform(1.0, 2.5) if aug else CANON_UPSCALE
    g2 = cv2.resize(g, (int(round(STORE_W * f)), int(round(STORE_H * f))), interpolation=cv2.INTER_CUBIC)
    fx, fy = g2.shape[1] / STORE_W, g2.shape[0] / STORE_H
    block = rng.uniform(1.8, 3.5) if aug else CANON_BLOCK_SPACES
    c = rng.uniform(6.0, 14.0) if aug else CANON_C
    b = binarize_like_stage1(g2, STORE_PPS * fy, block, c)
    if aug and rng.random() < 0.3:
        kernel = np.ones((2, 2), np.uint8)
        b = cv2.erode(b, kernel) if rng.random() < 0.5 else cv2.dilate(b, kernel)
    geo = {}
    if aug:
        geo = {"dx_spaces": rng.uniform(-GEO_DX, GEO_DX), "dy_spaces": rng.uniform(-GEO_DY, GEO_DY),
               "scale": rng.uniform(*GEO_SCALE), "angle_deg": rng.uniform(-GEO_ANGLE, GEO_ANGLE)}
    return ck.make_input(b, (x0_store + 0.5) * fx - 0.5, (STORE_TOP + 0.5) * fy - 0.5,
                         (STORE_BOTTOM + 0.5) * fy - 0.5, **geo)


class StaffStartDataset(Dataset):
    """Training set. The random stream of sample i in epoch e is seeded by (SEED, e, i)."""

    def __init__(self, indices: np.ndarray):
        self.indices = np.asarray(indices)
        self.epoch = 0

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = int(self.indices[k])
        rng = np.random.default_rng([SEED, self.epoch, i])
        x = build_input(RAW[i], float(X0S[i]), rng)
        return torch.from_numpy(x)[None], int(CLEF_Y[i]), int(KEY_Y[i])


def _canon_one(i):
    return build_input(RAW[i], float(X0S[i]), None).astype(np.float16)


def _stress_one(i):
    return build_input(RAW[i], float(X0S[i]), np.random.default_rng([SEED + 1000, 0, int(i)])).astype(np.float16)


def build_many(fn, idx):
    if len(idx) == 0:
        return np.zeros((0, ck.INPUT_H, ck.INPUT_W), np.float16)
    if N_PROC > 1:
        with ProcessPoolExecutor(max_workers=N_PROC) as ex:
            return np.stack(list(ex.map(fn, [int(i) for i in idx], chunksize=64)))
    return np.stack([fn(int(i)) for i in idx])


t0 = time.time()
EVAL_INPUTS = {name: build_many(_canon_one, idx) for name, idx in EVAL_SETS.items()}
print(f"clean eval inputs built in {time.time() - t0:.0f}s:", {k: len(v) for k, v in EVAL_INPUTS.items()})

## Step 13. Look at what the network will see

Top rows: training inputs with damage. Bottom row: clean validation inputs. White = ink. You should see
five staff lines across the middle, the clef near the left and the key signature after it. If the staff is
not centred, or the clef is cut off, stop and report it: the crop geometry is wrong.

In [ ]:
_ds_preview = StaffStartDataset(TRAIN_IDX[:: max(1, len(TRAIN_IDX) // 16)][:16])
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
for k, ax in enumerate(axes[:2].ravel()):
    if k < len(_ds_preview):
        x, c, kk = _ds_preview[k]
        ax.imshow(x[0].numpy(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{ck.CLEF_CLASSES[c]} key {ck.index_to_key(kk) if kk >= 0 else 'ign'}", fontsize=8)
    ax.axis("off")
_vi = EVAL_SETS["val_synth"][:4].tolist() + EVAL_SETS["val_ds2"][:4].tolist()
for ax, (j, i) in zip(axes[2], enumerate(_vi)):
    name = "val_synth" if j < min(4, len(EVAL_SETS["val_synth"])) else "val_ds2"
    pos = int(np.nonzero(EVAL_SETS[name] == i)[0][0])
    ax.imshow(EVAL_INPUTS[name][pos].astype(np.float32), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"val {SRC[i]} {ck.CLEF_CLASSES[CLEF_Y[i]]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(WORK / "preview_inputs.png", dpi=80)
plt.show()

## Step 14. Train

* Batches are drawn so that DS2 and synthetic staves each make up about half of every epoch, and rare clef
  classes (G8vb, other) are drawn more often (weight ~ 1/sqrt(class size)). The draw order comes from a
  seeded generator.
* Loss = clef cross-entropy + key cross-entropy (key rows marked "ignore" do not count for the key loss).
* After every epoch the model is checked on the **validation** sets only. The epoch with the best
  validation score (average over the two val sources of (clef balanced recall + key accuracy) / 2) is kept.
  Test sets are not looked at until Step 16.

In [ ]:
def sampler_weights(idx: np.ndarray) -> np.ndarray:
    w = np.zeros(len(idx), dtype=np.float64)
    for src in np.unique(SRC[idx]):
        m_src = SRC[idx] == src
        for c in np.unique(CLEF_Y[idx][m_src]):
            m = m_src & (CLEF_Y[idx] == c)
            w[m] = 1.0 / math.sqrt(m.sum())  # so the class's total weight grows like sqrt(its size)
        w[m_src] *= 0.5 / w[m_src].sum()
    return w


@torch.no_grad()
def logits_for(model, inputs: np.ndarray, batch: int = 512):
    model.eval()
    cl, kl = [], []
    for s in range(0, len(inputs), batch):
        x = torch.from_numpy(inputs[s:s + batch].astype(np.float32))[:, None].to(DEVICE)
        a, b = model(x)
        cl.append(a.float().cpu())
        kl.append(b.float().cpu())
    if not cl:
        return torch.zeros(0, len(ck.CLEF_CLASSES)), torch.zeros(0, len(ck.KEY_CLASSES))
    return torch.cat(cl), torch.cat(kl)


def quick_scores(model, name):
    idx = EVAL_SETS[name]
    if len(idx) == 0:
        return None
    cl, kl = logits_for(model, EVAL_INPUTS[name])
    yc, yk = CLEF_Y[idx], KEY_Y[idx]
    pc = cl.argmax(1).numpy()
    recalls = [float((pc[yc == c] == c).mean()) for c in range(len(ck.CLEF_CLASSES)) if (yc == c).any()]
    mk = yk != KEY_IGNORE
    key_acc = float((kl.argmax(1).numpy()[mk] == yk[mk]).mean()) if mk.any() else float("nan")
    return {"clef_acc": float((pc == yc).mean()), "clef_macro_recall": float(np.mean(recalls)), "key_acc": key_acc}


def worker_init(worker_id: int) -> None:
    cv2.setNumThreads(1)
    np.random.seed((SEED + worker_id) % (2 ** 32))
    random.seed(SEED + worker_id)


seed_everything(SEED)
model = ck.ClefKeyNet().to(DEVICE)
train_ds = StaffStartDataset(TRAIN_IDX)
sampler_gen = torch.Generator().manual_seed(SEED)
sampler = WeightedRandomSampler(torch.as_tensor(sampler_weights(TRAIN_IDX)), num_samples=len(TRAIN_IDX),
                                replacement=True, generator=sampler_gen)
loader_gen = torch.Generator().manual_seed(SEED + 1)
steps_per_epoch = max(1, len(TRAIN_IDX) // BATCH_SIZE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * steps_per_epoch)
clef_loss_fn = nn.CrossEntropyLoss()
key_loss_fn = nn.CrossEntropyLoss(ignore_index=KEY_IGNORE)

history, best = [], {"score": -1.0, "epoch": None, "state": None}
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_ds.epoch = epoch
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                        worker_init_fn=worker_init, generator=loader_gen, drop_last=True,
                        persistent_workers=False, pin_memory=DEVICE.type == "cuda")
    model.train()
    tot, nb = 0.0, 0
    for x, yc, yk in loader:
        x, yc, yk = x.to(DEVICE), yc.to(DEVICE), yk.to(DEVICE)
        cl, kl = model(x)
        loss = clef_loss_fn(cl, yc)
        if (yk != KEY_IGNORE).any():
            loss = loss + key_loss_fn(kl, yk)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        tot += float(loss)
        nb += 1
    val = {n: quick_scores(model, n) for n in VAL_SETS}
    parts = [0.5 * (v["clef_macro_recall"] + (v["key_acc"] if not math.isnan(v["key_acc"]) else 0.0))
             for v in val.values() if v is not None]
    score = float(np.mean(parts)) if parts else 0.0
    history.append({"epoch": epoch, "train_loss": tot / max(1, nb), "val": val, "val_score": score})
    if score > best["score"]:
        best = {"score": score, "epoch": epoch, "state": copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})}
    print(f"epoch {epoch:2d} loss {tot / max(1, nb):.4f} val_score {score:.4f} "
          + " | ".join(f"{n}: clef {v['clef_macro_recall']:.3f} key {v['key_acc']:.3f}" for n, v in val.items() if v)
          + f" ({time.time() - t0:.0f}s)")

model.load_state_dict(best["state"])
print("best epoch", best["epoch"], "val score", round(best["score"], 4))
(WORK / "clef_key_history.json").write_text(json.dumps(history, indent=1), encoding="utf-8")

## Step 15. Calibrate confidence and set the "unsure" thresholds (validation data only)

* **Temperature scaling**: one number per head, fitted on the validation sets, so that "90% confident"
  means roughly "right 90% of the time" on validation data.
* **Thresholds**: for each validation source separately, the lowest confidence at which the confident answers
  reach the pre-registered target (clef 99.5%, key 99%). The stricter of the two sources is used. Below the
  threshold, Stage 3 marks the staff `clef_unsure` / `key_unsure` for human review.
  If a target cannot be reached, the threshold is set to 1.01 (always unsure) and a warning is printed:
  that is a real result (the model is not good enough), not something to tune away.
* **G8vb decision**: G8vb is only trusted as a separate class if, on every validation source that has at
  least 30 true G8vb staves, recall >= 95% and <= 2% of plain G staves are called G8vb. Otherwise
  `g8vb_reliable=False` and Stage 3 treats G and G8vb as one "treble family" answer (voice decides the octave).

None of these use test data, and none are tuned by looking at test data afterwards.

In [ ]:
def fit_temperature(logits: torch.Tensor, y: torch.Tensor) -> float:
    if len(y) == 0:
        return 1.0
    log_t = torch.zeros(1, requires_grad=True)
    opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def closure():
        opt_t.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), y)
        loss.backward()
        return loss

    opt_t.step(closure)
    return float(log_t.exp().clamp(0.05, 20.0))


val_logits = {n: logits_for(model, EVAL_INPUTS[n]) for n in VAL_SETS}
all_cl = torch.cat([val_logits[n][0] for n in VAL_SETS])
all_kl = torch.cat([val_logits[n][1] for n in VAL_SETS])
all_yc = torch.as_tensor(np.concatenate([CLEF_Y[EVAL_SETS[n]] for n in VAL_SETS]))
all_yk = torch.as_tensor(np.concatenate([KEY_Y[EVAL_SETS[n]] for n in VAL_SETS]))
mk = all_yk != KEY_IGNORE
CLEF_T = fit_temperature(all_cl, all_yc)
KEY_T = fit_temperature(all_kl[mk], all_yk[mk])
print("temperatures: clef", round(CLEF_T, 3), "key", round(KEY_T, 3))

THRESH_GRID = np.concatenate([np.arange(0.30, 0.995, 0.005), [0.995, 0.998, 0.999]])


def lowest_threshold(conf: np.ndarray, correct: np.ndarray, target: float):
    for t in THRESH_GRID:
        m = conf >= t
        if m.sum() >= MIN_SELECTED_FOR_THRESHOLD and correct[m].mean() >= target:
            return float(t)
    return None


threshold_evidence = {}
clef_ts, key_ts = [], []
for n in VAL_SETS:
    idx = EVAL_SETS[n]
    if len(idx) == 0:
        continue
    pc = torch.softmax(val_logits[n][0] / CLEF_T, 1).numpy()
    pk = torch.softmax(val_logits[n][1] / KEY_T, 1).numpy()
    yc, yk = CLEF_Y[idx], KEY_Y[idx]
    # A confident "other" is still unsure for PiyaNota, so only in-scope predictions count as "sure".
    in_scope = pc.argmax(1) != ck.CLEF_CLASSES.index("other")
    tc = lowest_threshold(np.where(in_scope, pc.max(1), 0.0), pc.argmax(1) == yc, CLEF_SELECTIVE_TARGET)
    m = yk != KEY_IGNORE
    tk_ = lowest_threshold(pk.max(1)[m], pk.argmax(1)[m] == yk[m], KEY_SELECTIVE_TARGET) if m.any() else None
    threshold_evidence[n] = {"clef_threshold": tc, "key_threshold": tk_, "n": int(len(idx)), "n_key": int(m.sum())}
    clef_ts.append(1.01 if tc is None else tc)
    key_ts.append(1.01 if tk_ is None else tk_)
CLEF_THRESHOLD = float(max(clef_ts)) if clef_ts else 1.01
KEY_THRESHOLD = float(max(key_ts)) if key_ts else 1.01
for name, t in (("clef", CLEF_THRESHOLD), ("key", KEY_THRESHOLD)):
    if t > 1.0:
        print(f"WARNING: the {name} target was not reached on some validation source: every {name} answer will "
              "be marked unsure. Report this; do not lower the target to make it pass.")
print("thresholds: clef", CLEF_THRESHOLD, "key", KEY_THRESHOLD, threshold_evidence)

G, G8 = ck.CLEF_CLASSES.index("G"), ck.CLEF_CLASSES.index("G8vb")
g8vb_evidence = {}
for n in VAL_SETS:
    idx = EVAL_SETS[n]
    yc = CLEF_Y[idx]
    pred = val_logits[n][0].argmax(1).numpy() if len(idx) else np.zeros(0, int)
    n8, ng = int((yc == G8).sum()), int((yc == G).sum())
    g8vb_evidence[n] = {"n_g8vb": n8, "n_g": ng,
                        "g8vb_recall": float((pred[yc == G8] == G8).mean()) if n8 else None,
                        "g_called_g8vb": float((pred[yc == G] == G8).mean()) if ng else None}
qualifying = [e for e in g8vb_evidence.values() if e["n_g8vb"] >= G8VB_MIN_SUPPORT and e["n_g"] > 0]
G8VB_RELIABLE = bool(qualifying) and all(e["g8vb_recall"] >= G8VB_MIN_RECALL
                                         and e["g_called_g8vb"] <= G8VB_MAX_G_CONFUSION for e in qualifying)
print("g8vb_reliable =", G8VB_RELIABLE, g8vb_evidence)

## Step 16. Final evaluation on the held-out test sets (looked at once)

Every set is reported on its own line; nothing is averaged across sources. For each:
* clef accuracy with a 95% confidence interval (Wilson), balanced recall, per-class confusion matrix;
* key accuracy (on staves with a usable key label) with its interval, and how often it is off by one;
* **confident-and-wrong rate**: the fraction of all staves where the model was above the threshold and
  still wrong. This is the BUG-011 failure mode (the CRNN was confidently wrong). It must be small;
* AUROC of confidence for right-vs-wrong (1.0 = confidence perfectly separates right from wrong,
  0.5 = useless; the CRNN's clef confidence was 0.64);
* ECE (calibration error; lower is better);
* the same on a *stress* copy with training-style damage.

These are synthetic and DS2 numbers. They are **not** PiyaNota accuracy. Real accuracy has to be measured
later on real photos (see the plan doc, validation plan), and the test photos stay untouched until then.

In [ ]:
def wilson(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (None, None)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def auroc(score: np.ndarray, positive: np.ndarray):
    pos, neg = score[positive], score[~positive]
    if len(pos) == 0 or len(neg) == 0:
        return None
    order = np.argsort(np.concatenate([pos, neg]), kind="mergesort")
    ranks = np.empty(len(order))
    ranks[order] = np.arange(1, len(order) + 1)
    allv = np.concatenate([pos, neg])
    for v in np.unique(allv):  # average ranks for ties
        m = allv == v
        ranks[m] = ranks[m].mean()
    return float((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


def ece(conf: np.ndarray, correct: np.ndarray, bins: int = 15):
    if len(conf) == 0:
        return None
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


def evaluate(name: str, idx: np.ndarray, inputs: np.ndarray):
    if len(idx) == 0:
        return {"n": 0}
    cl, kl = logits_for(model, inputs)
    pc = torch.softmax(cl / CLEF_T, 1).numpy()
    pk = torch.softmax(kl / KEY_T, 1).numpy()
    yc, yk = CLEF_Y[idx], KEY_Y[idx]
    pred_c, conf_c = pc.argmax(1), pc.max(1)
    ok_c = pred_c == yc
    other = ck.CLEF_CLASSES.index("other")
    sure_c = (conf_c >= CLEF_THRESHOLD) & (pred_c != other)
    fam = lambda a: np.where(a == G8, G, a)
    k_c = int(ok_c.sum())
    res = {"n": int(len(idx)),
           "clef_acc": k_c / len(idx), "clef_acc_ci95": wilson(k_c, len(idx)),
           "clef_acc_treble_family": float((fam(pred_c) == fam(yc)).mean()),
           "clef_macro_recall": float(np.mean([(pred_c[yc == c] == c).mean()
                                               for c in range(len(ck.CLEF_CLASSES)) if (yc == c).any()])),
           "clef_confusion_rows_true_cols_pred": [[int(((yc == a) & (pred_c == b)).sum())
                                                   for b in range(len(ck.CLEF_CLASSES))]
                                                  for a in range(len(ck.CLEF_CLASSES))],
           "clef_coverage_sure": float(sure_c.mean()),
           "clef_acc_when_sure": float(ok_c[sure_c].mean()) if sure_c.any() else None,
           "clef_confident_wrong_rate": float((sure_c & ~ok_c).mean()),
           "clef_auroc": auroc(conf_c, ok_c), "clef_ece": ece(conf_c, ok_c)}
    m = yk != KEY_IGNORE
    if m.any():
        pred_k, conf_k = pk.argmax(1)[m], pk.max(1)[m]
        ok_k = pred_k == yk[m]
        sure_k = conf_k >= KEY_THRESHOLD
        res.update({"n_key": int(m.sum()), "key_acc": float(ok_k.mean()),
                    "key_acc_ci95": wilson(int(ok_k.sum()), int(m.sum())),
                    "key_off_by_one": float((np.abs(pred_k - yk[m]) == 1).mean()),
                    "key_coverage_sure": float(sure_k.mean()),
                    "key_acc_when_sure": float(ok_k[sure_k].mean()) if sure_k.any() else None,
                    "key_confident_wrong_rate": float((sure_k & ~ok_k).mean()),
                    "key_auroc": auroc(conf_k, ok_k), "key_ece": ece(conf_k, ok_k)})
    return res


TEST_SETS = [n for n in EVAL_SETS if n.startswith("test")]
METRICS = {"val": {n: evaluate(n, EVAL_SETS[n], EVAL_INPUTS[n]) for n in VAL_SETS}, "test": {}, "stress": {}}
for n in TEST_SETS:
    METRICS["test"][n] = evaluate(n, EVAL_SETS[n], EVAL_INPUTS[n])
    METRICS["stress"][n] = evaluate(n, EVAL_SETS[n], build_many(_stress_one, EVAL_SETS[n]))


def _fmt(v):
    return "-" if v is None else (f"{v:.3f}" if isinstance(v, float) else str(v))


print(f"{'set':44s} {'n':>5s} {'clef':>6s} {'ci_lo':>6s} {'bal':>6s} {'cov':>6s} {'cWrong':>7s} {'cAUC':>6s}"
      f" {'key':>6s} {'kWrong':>7s} {'kAUC':>6s}")
for group in ("val", "test", "stress"):
    for n, r in METRICS[group].items():
        if r["n"] == 0:
            print(f"{group + ':' + n:44s} (empty)")
            continue
        print(f"{group + ':' + n:44s} {r['n']:5d} {_fmt(r['clef_acc']):>6s} {_fmt(r['clef_acc_ci95'][0]):>6s} "
              f"{_fmt(r['clef_macro_recall']):>6s} {_fmt(r['clef_coverage_sure']):>6s} "
              f"{_fmt(r['clef_confident_wrong_rate']):>7s} {_fmt(r['clef_auroc']):>6s} "
              f"{_fmt(r.get('key_acc')):>6s} {_fmt(r.get('key_confident_wrong_rate')):>7s} {_fmt(r.get('key_auroc')):>6s}")
for n, r in METRICS["test"].items():
    if r["n"]:
        print(n, "clef confusion (rows = true", ck.CLEF_CLASSES, "):", r["clef_confusion_rows_true_cols_pred"])

## Step 17. Save the weights, the metadata and a parity fixture

* `clef_key_classifier.pt`: weights **plus** everything Stage 3 needs to use them correctly (class names,
  crop settings, temperatures, thresholds, the G8vb decision, seed, split-manifest hash, metrics, versions).
* `clef_key_parity_fixture.npz`: a few binarised test crops with the exact inputs and outputs produced here.
  A repo test re-runs `make_input` + the model on them; if the numbers differ, the repo's crop code or the
  weights file does not match this run.
* The file is reloaded into a fresh model and checked before the cell finishes.

In [ ]:
fixture_idx = (EVAL_SETS["test_ds2_seen_fonts"][:8].tolist() + EVAL_SETS["test_synth_seen_fonts"][:8].tolist())
fx_bin, fx_geom, fx_inputs = [], [], []
for i in fixture_idx:
    g2 = cv2.resize(RAW[i], (int(round(STORE_W * CANON_UPSCALE)), int(round(STORE_H * CANON_UPSCALE))),
                    interpolation=cv2.INTER_CUBIC)
    fx, fy = g2.shape[1] / STORE_W, g2.shape[0] / STORE_H
    b = binarize_like_stage1(g2, STORE_PPS * fy, CANON_BLOCK_SPACES, CANON_C)
    geom = ((float(X0S[i]) + 0.5) * fx - 0.5, (STORE_TOP + 0.5) * fy - 0.5, (STORE_BOTTOM + 0.5) * fy - 0.5)
    fx_bin.append(b)
    fx_geom.append(geom)
    fx_inputs.append(ck.make_input(b, *geom))
fx_inputs = np.stack(fx_inputs) if fx_inputs else np.zeros((0, ck.INPUT_H, ck.INPUT_W), np.float32)
fx_cl, fx_kl = logits_for(model, fx_inputs)
np.savez_compressed(WORK / "clef_key_parity_fixture.npz",
                    binary=np.stack(fx_bin) if fx_bin else np.zeros((0, 1, 1), np.uint8),
                    geometry=np.array(fx_geom, dtype=np.float64).reshape(-1, 3), inputs=fx_inputs,
                    clef_logits=fx_cl.numpy(), key_logits=fx_kl.numpy(),
                    clef_labels=CLEF_Y[fixture_idx], key_labels=KEY_Y[fixture_idx])

data_summary = {r[0]: {"n": r[1], **{c: v for c, v in zip(ck.CLEF_CLASSES, r[2:6])}, "key_labelled": r[6]}
                for r in rows}
checkpoint = {
    "model_state": {k: v.cpu() for k, v in model.state_dict().items()},
    "arch": {"name": "ClefKeyNet", "width": 32, "dropout": 0.3},
    "crop_spec_version": ck.CROP_SPEC_VERSION,
    "crop_spec": {"px_per_space": ck.PX_PER_SPACE, "left": ck.CROP_LEFT_SPACES, "right": ck.CROP_RIGHT_SPACES,
                  "above": ck.CROP_ABOVE_SPACES, "below": ck.CROP_BELOW_SPACES,
                  "input_h": ck.INPUT_H, "input_w": ck.INPUT_W},
    "core_sha256": CORE_SHA256,
    "input_contract": ("Binary page (uint8, ink=0, paper=255) = Stage 1 native adaptive-threshold strip source "
                       "BEFORE non_staff_ink_mask (PreprocessedPage.strip_unmasked). x0 = staff_left_edge(); "
                       "top/bottom = centre rows of the outer staff lines in the same pixel grid."),
    "clef_classes": list(ck.CLEF_CLASSES), "key_classes": list(ck.KEY_CLASSES),
    "clef_temperature": CLEF_T, "key_temperature": KEY_T,
    "clef_threshold": CLEF_THRESHOLD, "key_threshold": KEY_THRESHOLD,
    "clef_selective_target": CLEF_SELECTIVE_TARGET, "key_selective_target": KEY_SELECTIVE_TARGET,
    "threshold_evidence_val": threshold_evidence,
    "g8vb_reliable": G8VB_RELIABLE, "g8vb_evidence_val": g8vb_evidence,
    "g8vb_rule": {"min_recall": G8VB_MIN_RECALL, "max_g_confusion": G8VB_MAX_G_CONFUSION,
                  "min_support": G8VB_MIN_SUPPORT},
    "seed": SEED, "best_epoch": best["epoch"], "epochs": EPOCHS, "best_val_score": best["score"],
    "selection": "best val_score (mean over val_ds2/val_synth of (clef balanced recall + key acc)/2); "
                 "temperatures and thresholds fitted on val only",
    "split_manifest_sha256": SPLIT_MANIFEST_SHA256,
    "metrics": METRICS, "data_summary": data_summary, "versions": VERSIONS,
    "synthetic": {"n_systems": N_SYNTH_SYSTEMS, "train_fonts": list(SYNTH_TRAIN_FONTS),
                  "unseen_font": SYNTH_UNSEEN_FONT, "status": dict(synth_status)},
    "ds2_unseen_font": DS2_UNSEEN_FONT,
    "never_used": "piyanota_dataset (incl. Flashlight), DSMN, Both Sides Now: not used for training, "
                  "selection, calibration or thresholds",
    "smoke_test": SMOKE_TEST,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
ckpt_path = WORK / "clef_key_classifier.pt"
torch.save(checkpoint, ckpt_path)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(type(o))


meta_out = {k: v for k, v in checkpoint.items() if k != "model_state"}
(WORK / "clef_key_metrics.json").write_text(json.dumps(meta_out, indent=1, default=_json_default), encoding="utf-8")

# Reload check: a fresh model from the saved file must give the same logits.
reloaded = torch.load(ckpt_path, map_location="cpu", weights_only=False)
fresh = ck.ClefKeyNet(width=reloaded["arch"]["width"], dropout=reloaded["arch"]["dropout"])
fresh.load_state_dict(reloaded["model_state"])
fresh.to(DEVICE).eval()
with torch.no_grad():
    rc, rk = fresh(torch.from_numpy(fx_inputs.astype(np.float32))[:, None].to(DEVICE))
assert torch.allclose(rc.cpu(), fx_cl, atol=1e-4) and torch.allclose(rk.cpu(), fx_kl, atol=1e-4), "reload mismatch"
print("saved and reload-checked:", ckpt_path, "smoke_test =", SMOKE_TEST)

## Step 18. What to download and where it goes

The list printed below comes from `/kaggle/working`. After a **Save & Run All (Commit)** run, open the
notebook's page -> **Output** tab (or the version's *Output* section) and download these files. Put them in
the repo exactly here (paths are for the `D:\PiyaNota-training` worktree; use the same relative paths in
any other checkout):

| File | Put it at | Why |
|---|---|---|
| `clef_key_classifier.pt` | `music_symbol_detection\clef_key_classifier.pt` | the weights + metadata (Git LFS tracks `*.pt`) |
| `clef_key_core.py` | `clef_key_core.py` (repo root, next to `stage3_transcription.py`) | the crop + network code Stage 3 must import unchanged |
| `clef_key_metrics.json` | `training\clef_key_artifacts\clef_key_metrics.json` | all numbers from Steps 15-16 |
| `clef_key_splits.json` | `training\clef_key_artifacts\clef_key_splits.json` | proof of which pieces were held out |
| `clef_key_history.json` | `training\clef_key_artifacts\clef_key_history.json` | per-epoch validation curve |
| `clef_key_parity_fixture.npz` | `training\clef_key_artifacts\clef_key_parity_fixture.npz` | repo-side crop/weights check |
| `preview_inputs.png`, `preview_synthetic_systems.png` | `training\clef_key_artifacts\` | for the review |

Do not rename the files. Do not commit anything yet: the metrics go to ml-cv-lead for review first, and if
the run was a `SMOKE_TEST` run, throw the weights away.
Also note the Kaggle notebook **version number** you downloaded from, for the decision record.

In [ ]:
print("SMOKE TEST RUN - weights are NOT usable" if SMOKE_TEST else "Full run.")
for p in sorted(WORK.iterdir()):
    if p.is_file() and (p.name.startswith("clef_key") or p.name.startswith("preview_")):
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        print(f"{p.name:36s} {p.stat().st_size / 1e6:8.2f} MB  sha256 {h[:16]}")
print("best epoch", best["epoch"], "| thresholds clef", CLEF_THRESHOLD, "key", KEY_THRESHOLD,
      "| g8vb_reliable", G8VB_RELIABLE)